# GRN Construction (FlowGRN)

Alternative to **"5. GRN Construction.ipynb"**: instead of a curated CollecTRI PKN, builds a **data-driven** PKN from expression trajectories via FlowGRN (flow-matching trajectory reconstruction + dynGENIE3), then thresholds it and looks up curated signs from CollecTRI where available (falling back to `sign=0` for edges FlowGRN discovered that aren't in CollecTRI — bonesis lets the solver pick the monotonicity from the data).

Ends by exporting the same 4-file bundle as "5. GRN Construction.ipynb" (`{FILE_PREFIX}_prior_grn.json`, `{FILE_PREFIX}_pkn.json`, `{FILE_PREFIX}_data_meta.json`, `{FILE_PREFIX}_config.json`), consumed unchanged by **"6. BN Inference (BoNesis).ipynb"**.

**Requirements before this notebook can actually run:**
- `results/{METHOD}/trajectory.h5ad` for the *retina* dataset (produced by "4. Trajectory Inference.ipynb") — the only one currently on disk (`results/scvelo_stochastic/trajectory.h5ad`) is for a different dataset (Pancreas), so `TRAJECTORY_H5AD` below is a placeholder.
- `GSE122466_Retina_vargenes_batch1.csv` (same missing file flagged in "5. GRN Construction.ipynb").
- A completed run of **"4.5. Phenotype Observations.ipynb"** for `SOURCE_FILE_PREFIX` (its `meta`/`config` export is reused here — the binarized phenotype observations don't depend on which PKN source is used downstream. Note: this is `meta`, the *unfiltered* phenotype x gene matrix — not "5. GRN Construction.ipynb"'s `data_meta`, which is already restricted to CollecTRI genes and would wrongly drop genes FlowGRN discovers outside CollecTRI).
- The `flowgrn` conda env (`envs/flowgrn.yaml`) installed as a sibling of this kernel's env.

This notebook itself runs on the `scBoolSeq` kernel (like NB5/NB6) — only trajectory reconstruction (torch/torchcfm) is shelled out to the `flowgrn` env via subprocess; dynGENIE3 itself is pure sklearn/numpy and runs in-process.

In [ ]:
import sys
import json
from pathlib import Path as path

import numpy as np
import pandas as pd
import networkx as nx
import anndata as ad
import bonesis

# Repo root → src on path (shared with the RNA velocity / trajectory-inference notebooks)
ROOT = path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from grn_inference import (
    load_collectri,
    derive_branches,
    write_beeline_inputs,
    run_trajectory_reconstruction,
    concatenate_trajectories,
    run_dyngenie3,
    threshold_vim_topk,
    edges_to_signed_graph,
    df_to_bonesis_data,
    export_pkn_bundle,
)


In [ ]:
FILE_PREFIX = "flowgrn"  # output filename prefix for this notebook's PKN bundle

# Which prior "4.5. Phenotype Observations.ipynb" run's meta/config to reuse
# (the binarized phenotype observations don't depend on the PKN source).
SOURCE_FILE_PREFIX = "sum"

# Placeholder — adjust once retina trajectory inference (NB "4. Trajectory Inference")
# has actually been run; the only trajectory.h5ad on disk right now is for Pancreas.
TRAJECTORY_H5AD = "results/paga_dpt/trajectory.h5ad"

EXPR_CSV = "GSE122466_Retina_vargenes_batch1.csv"  # same expression matrix as NB5

FLOWGRN_REPO = ROOT / "FlowGRN"
# flowgrn env assumed to be a sibling of this kernel's own conda env
FLOWGRN_PYTHON = path(sys.exec_prefix).parent / "flowgrn" / "bin" / "python"

N_SNAPSHOT = 5   # FlowGRN default; time bins per branch for trajectory reconstruction
TOP_K = 10       # regulators kept per target gene when thresholding VIM
NTREES = 1000    # dynGENIE3 default
SEED = 0

print(f"FILE_PREFIX={FILE_PREFIX!r}, SOURCE_FILE_PREFIX={SOURCE_FILE_PREFIX!r}")
print(f"FLOWGRN_PYTHON={FLOWGRN_PYTHON}")


### TF database (for candidate regulators + sign lookup)

In [ ]:
net = load_collectri()
net.head()


### Trajectory contract → branches

In [ ]:
ti = ad.read_h5ad(TRAJECTORY_H5AD)
print(f"root_state={ti.uns['root_state']!r}, terminal_states={list(ti.uns['terminal_states'])}")
ti.obs[["label", "pseudotime"]].head()


In [ ]:
branches = derive_branches(ti)
branches


In [ ]:
data = pd.read_csv(EXPR_CSV, index_col=0)  # cells x genes, same file as NB5
print(data.shape)
data.head()


In [ ]:
bundle = write_beeline_inputs(ti, data, branches, out_dir=f"{FILE_PREFIX}_beeline_inputs")
bundle


### Trajectory reconstruction (subprocess, `flowgrn` env)

In [ ]:
%%time
traj_paths = []
for branch_id, terminal in enumerate(bundle["branch_order"]):
    print(f"Reconstructing branch {branch_id} ({terminal})...")
    traj_paths.append(
        run_trajectory_reconstruction(
            bundle["expr_fn"],
            bundle["pseudoT_fn"],
            branch_id,
            out_dir=f"{FILE_PREFIX}_traj_out",
            flowgrn_repo=FLOWGRN_REPO,
            flowgrn_python=FLOWGRN_PYTHON,
            seed=SEED,
            n_snapshot=N_SNAPSHOT,
        )
    )
traj_paths


In [ ]:
traj = concatenate_trajectories(traj_paths)
traj.shape  # (n_time, total_n_cell, n_gene)


### Network inference (dynGENIE3, in-process)

In [ ]:
# Candidate regulators restricted to known CollecTRI TFs present in this expression matrix
tf_candidates = sorted(set(net["source"]) & set(data.columns))
len(tf_candidates)


In [ ]:
%%time
vim = run_dyngenie3(
    traj,
    gene_names=list(data.columns),
    regulators=tf_candidates,
    flowgrn_repo=FLOWGRN_REPO,
    ntrees=NTREES,
)
vim.shape


In [ ]:
edges = threshold_vim_topk(vim, k=TOP_K)
print(edges.shape)
edges.head()


In [ ]:
flowgrn_grn = edges_to_signed_graph(edges, net)
print(f"nodes={len(flowgrn_grn)}, edges={flowgrn_grn.number_of_edges()}")
nx.is_weakly_connected(flowgrn_grn)


### Reuse phenotype observations from "4.5. Phenotype Observations.ipynb"

In [ ]:
meta = pd.read_csv(f"meta_{SOURCE_FILE_PREFIX}_bin_aggregated.csv", index_col=0)

with open(f"{SOURCE_FILE_PREFIX}_config.json") as _f:
    INCLUDE_HC_CONSTRAINT = json.load(_f)["INCLUDE_HC_CONSTRAINT"]

# Restrict to genes present in the FlowGRN-derived PKN (BoNesis needs data_meta's
# genes to be a subset of pkn's nodes) — using the *unfiltered* meta here (not NB5's
# CollecTRI-restricted data_meta) so genes FlowGRN discovers outside CollecTRI aren't
# silently dropped.
flowgrn_nodes = set(flowgrn_grn.nodes)
meta_flowgrn = meta[[g for g in meta.columns if g in flowgrn_nodes]]
data_meta = df_to_bonesis_data(meta_flowgrn)

print(f"INCLUDE_HC_CONSTRAINT={INCLUDE_HC_CONSTRAINT}")
data_meta.keys()


### BoNesis PKN + export

In [ ]:
# No separate biggest_wcc variant here (unlike NB5) — if bonesis complains about
# disconnected components, `extract_largest_wcc(flowgrn_grn)` (from grn_inference)
# is a drop-in fix.
pkn = bonesis.domains.InfluenceGraph(
    flowgrn_grn, maxclause=8, allow_skipping_nodes=True, canonic=False
)
bo = bonesis.BoNesis(pkn, data_meta)
bo


In [ ]:
export_pkn_bundle(FILE_PREFIX, flowgrn_grn, pkn, data_meta, INCLUDE_HC_CONSTRAINT)

print(
    f"Exported prior_grn ({len(flowgrn_grn)} nodes), pkn ({len(pkn)} nodes), "
    f"data_meta ({len(data_meta)} observations) and config for FILE_PREFIX={FILE_PREFIX!r}"
)


### Next step

Continue in **"6. BN Inference (BoNesis).ipynb"** — set `FILE_PREFIX = "flowgrn"` there to load the files exported here.